In [3]:
pip install langchain langchain-groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.8 MB/s eta 0:00:00


In [8]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [9]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model = "llama-3.3-70b-versatile",temperature=0
)

In [10]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    "Explain {topic} in simple English."
)

In [14]:
formatted_prompt = prompt.invoke({
    "topic": "RAG"
})

print(formatted_prompt)

text='Explain RAG in simple English.'


In [16]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [18]:
prompt = PromptTemplate.from_template(
    "Explain {topic} in {language} for a {level} student."
)

In [19]:
formatted_prompt = prompt.invoke({
    "topic": "RAG",
    "language": "English",
    "level": "beginner"
})

print(formatted_prompt)

text='Explain RAG in English for a beginner student.'


In [21]:
from langchain_core.prompts import ChatPromptTemplate

In [23]:
chat_prompt = ChatPromptTemplate([
    ("system", "You are an expert AI teacher."),
    ("human", "Explain {topic} in simple English.")
])

In [24]:
messages = chat_prompt.invoke({
    "topic":"RAG"
})
print(messages)

messages=[SystemMessage(content='You are an expert AI teacher.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain RAG in simple English.', additional_kwargs={}, response_metadata={})]


In [26]:
chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful AI assistant. "
        "Answer only using the provided context."
    ),
    (
        "human",
        "Context:\n{context}\n\nQuestion:\n{question}"
    )
])

In [27]:
messages = chat_prompt.invoke({
    "context": """
    RAG stands for Retrieval-Augmented Generation.
    It retrieves relevant information before generating an answer.
    """,
    "question": "What does RAG stand for?"
})

In [34]:
#Retriever
from langchain_core.documents import Document

documents = [
    Document(
        page_content=(
            "The payment service experienced HTTP 504 errors "
            "because its database connection pool was exhausted. "
            "All 30 connections were active and none were idle."
        ),
        metadata={"source": "incident_001", "service": "payment-service"}
    ),
    Document(
        page_content=(
            "High database query latency can cause connections "
            "to remain occupied longer than expected. "
            "Slow queries should be investigated."
        ),
        metadata={"source": "incident_002", "service": "database"}
    ),
    Document(
        page_content=(
            "HTTP 504 indicates that a gateway did not receive "
            "a timely response from an upstream server."
        ),
        metadata={"source": "http_guide", "service": "gateway"}
    ),
    Document(
        page_content=(
            "Memory leaks occur when an application retains memory "
            "that it no longer needs."
        ),
        metadata={"source": "memory_guide", "service": "application"}
    )
]

print("Total documents:", len(documents))

Total documents: 4


In [37]:
!pip -q install -U langchain-huggingface

In [38]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    documents,
    embeddings
)

print("Vector store created successfully.")

/tmp/ipykernel_766/1091041161.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully.


In [39]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

In [40]:
question = "Why did the payment service return HTTP 504 errors?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nDocument {i}")
    print("Source:", doc.metadata["source"])
    print("Content:", doc.page_content)


Document 1
Source: incident_001
Content: The payment service experienced HTTP 504 errors because its database connection pool was exhausted. All 30 connections were active and none were idle.

Document 2
Source: http_guide
Content: HTTP 504 indicates that a gateway did not receive a timely response from an upstream server.


In [43]:
#tool

from langchain_core.tools import tool

@tool
def multiply(a:int,b:int) -> int:
  "Multiply two integers and return the result."
  return a*b

In [44]:
print("Name:", multiply.name)
print("Description:", multiply.description)
print("Schema:", multiply.args)

Name: multiply
Description: Multiply two integers and return the result.
Schema: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [45]:
result = multiply.invoke({
    "a": 125,
    "b": 48
})
print(result)

6000


In [46]:
tools = [multiply]

model_with_tools = model.bind_tools(tools)

In [49]:
#pydantic schema
from pydantic import BaseModel, Field
class IncidentReport(BaseModel):
    service: str = Field(description="Affected service name")
    severity: str = Field(description="Incident severity, if known")
    root_cause: str = Field(description="Likely cause based on the input")
    recommended_actions: list[str] = Field(
        description="Recommended investigation steps"
    )

In [51]:
from langchain_core.chat_history import InMemoryChatMessageHistory
history = InMemoryChatMessageHistory()
history.add_user_message("My name is Alex.")
history.add_ai_message("Nice to meet you, Alex.")
print(history.messages)

[HumanMessage(content='My name is Alex.', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Alex.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


/tmp/ipykernel_766/4188671275.py:2: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  history = InMemoryChatMessageHistory()


In [52]:
for message in history.messages:
  print(type(message).__name__, ":", message.content)

HumanMessage : My name is Alex.
AIMessage : Nice to meet you, Alex.


In [53]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use conversation history when relevant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

base_chain = prompt | model | StrOutputParser()

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [55]:
#Build an agent using Groq

!pip -q install -U langchain langchain-groq

In [56]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the exact result."""
    return a * b

@tool
def get_service_status(service: str) -> str:
    """Return a sample status for a known service."""
    sample_statuses = {
        "payment-service": "Degraded",
        "user-service": "Healthy"
    }

    return sample_statuses.get(
        service.lower(),
        "Unknown service"
    )

In [60]:
#create agent
from langchain.agents import create_agent
tools = [multiply,  get_service_status]
agent = create_agent(
    model = model,
    tools = tools,
    system_prompt=(
        "You are a technical support assistant. "
        "Use available tools when they help answer the question. "
        "Never claim that a tool checked a real system when it used sample data."
    )
)